In [10]:
!pip3 install autogen openai python-dotenv --break-system-packages

In [11]:
import warnings

# Suppress autogen and other deprecation/user warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)

from autogen import ConversableAgent, GroupChat, GroupChatManager
from openai import OpenAI
import logging

# Suppress warnings from autogen.oai.client
logging.getLogger("autogen.oai.client").setLevel(logging.ERROR)

### What is AutoGen?

AutoGen is an open-source framework from Microsoft for building multi-agent AI systems. Instead of a single model working alone, multiple agents collaborate, exchange outputs, and iteratively improve results for complex tasks.

##### Key Features of AutoGen

1. Multi-Agent Collaboration: Agents can take specialized roles (for example, generator, reviewer, tester) and work together.
2. Conversational and Task-Oriented AI: Supports multi-turn agent conversations to refine answers and route tasks.
3. Automated Workflow Generation: Enables dynamic workflows for coding, research, and document automation.
4. Human-AI Collaboration: Humans can intervene, guide, or approve steps in agent workflows.

##### Comparison: AutoGen vs Traditional AI Agents

<table width="60%" align="left" >
        <thead>
            <tr>
                <th>Feature</th>
                <th>Traditional AI Agents</th>
                <th>AutoGen AI Agents</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td>Interactivity</td>
                <td>Works alone</td>
                <td>Collaborates with other agents</td>
            </tr>
            <tr>
                <td>Learning Ability</td>
                <td>Static</td>
                <td>Adaptive & iterative</td>
            </tr>
            <tr>
                <td>Workflow Handling</td>
                <td>Predefined</td>
                <td>Dynamic & evolving</td>
            </tr>
            <tr>
                <td>Human Intervention</td>
                <td>Limited</td>
                <td>Supports human-AI collaboration</td>
            </tr>
        </tbody>
</table>

### How AutoMed Works: Multi-Agent AI in Action

AutoMed handles each user query through a coordinated team of specialized AI agents, not a single response generator. Agents collaborate in real time to analyze, validate, and improve recommendations for better accuracy and reliability.

For a user reporting persistent headaches and fatigue, AutoMed activates:

1. **Patient Agent**: collects symptoms, medical history, and current treatments.  
2. **Symptom Analyzer Agent**: evaluates likely causes (for example, migraine, dehydration, anemia).  
3. **Pharmacy Agent**: suggests suitable OTC options and safety guidance.  
4. **Consultation Advisor Agent**: determines whether clinical care is needed, using current trusted medical sources.

This role-based workflow produces more personalized and balanced guidance, similar to input from multiple healthcare professionals.

In [19]:
import os
from dotenv import load_dotenv

load_dotenv()

# Disable Docker execution to prevent runtime errors
code_execution_config = {"use_docker": False}

# Sample LLM Configuration (Replace with actual API keys/config if needed)
llm_config = {
    "config_list": [
        {
            "api_type": "openai",
            "model": "Qwen/Qwen3.5-397B-A17B:novita",
            # "model": "Intelligent-Internet/II-Medical-8B:featherless-ai",  # or your HF model/provider
            "api_key": os.getenv("HUGGINGFACEHUB_API_TOKEN") or os.getenv("HF_TOKEN"),
            "base_url": os.getenv("HUGGINGFACE_BASE_URL", "https://router.huggingface.co/v1"),
        }
    ],
    "temperature": 0,
    "timeout": 120,
}

##### What is `ConversableAgent`?

`ConversableAgent` is an AI agent that can participate in structured conversations using an LLM and a role defined by its system message.

In this setup:
- `patient_agent`: provides symptoms and requests help (input-focused).  
- `diagnosis_agent`: analyzes symptoms and returns a concise diagnosis.  
- `pharmacy_agent`: gives medication recommendations in a single response.  
- `consultation_agent`: decides whether doctor follow-up is needed, summarizes next steps, and ends the flow using `CONSULTATION_COMPLETE`.  

All agents share `llm_config` so responses are generated consistently within the same model setup.  
This creates a clear, role-based pipeline from symptom intake to recommendation and final consultation outcome.

In [13]:

# Step 1: Create AI Agents with Defined Roles
patient_agent = ConversableAgent(
    name="patient", 
    system_message="You describe symptoms and ask for medical help.", 
    llm_config=llm_config
)

diagnosis_agent = ConversableAgent(
    name="diagnosis", 
    system_message="You analyze symptoms and provide a possible diagnosis. Summarize key points in one response.", 
    llm_config=llm_config
)

pharmacy_agent = ConversableAgent(
    name="pharmacy", 
    system_message="You recommend medications based on diagnosis. Only respond once.", 
    llm_config=llm_config
)

consultation_agent = ConversableAgent(
    name="consultation", 
    system_message="You determine if a doctor's visit is required. Provide a final summary with clear next steps. IMPORTANT: End your response with 'CONSULTATION_COMPLETE' to signal the end of the conversation.", 
    llm_config=llm_config
)

##### What is `GroupChat`?

`GroupChat` manages structured multi-agent conversations with controlled turn-taking and stop limits.

In this setup:
- `agents=[diagnosis_agent, pharmacy_agent, consultation_agent]`: defines participating responders by role.  
- `messages=[]`: starts with a fresh, empty conversation state.  
- `max_round=5`: caps interaction cycles to avoid endless loops.  
- `speaker_selection_method="round_robin"`: enforces orderly, turn-based responses.

The patient agent starts the interaction but does not need to be part of the speaking rotation.

In [14]:
# Step 2: Create GroupChat for Structured Interaction
groupchat = GroupChat(
    agents=[diagnosis_agent, pharmacy_agent, consultation_agent],  # Patient only initiates
    messages=[], 
    max_round=5,  # Limits conversation to 5 rounds
    speaker_selection_method="round_robin"  # Ensures structured conversation flow
)

# Step 3: Create GroupChatManager to Handle Conversation
manager = GroupChatManager(name="manager", groupchat=groupchat)

In [15]:
# Step 4: Get Patient Input and Start Consultation
print("\n🤖 Welcome to the AI Healthcare Consultation System!")
symptoms = input("🩺 Please describe your symptoms: ")

print("\n🩺 Diagnosing symptoms...")
response = patient_agent.initiate_chat(
    manager, 
    message=f"I am feeling {symptoms}. Can you help?",
)


🤖 Welcome to the AI Healthcare Consultation System!

🩺 Diagnosing symptoms...
patient (to manager):

I am feeling headache, fever, sneezing, coughing, muscle aches. Can you help?

--------------------------------------------------------------------------------

Next speaker: diagnosis


>>>>>>>> USING AUTO REPLY...
diagnosis (to manager):

<Answer>**Possible Diagnosis:**  
- **Influenza (Flu):** Matches symptoms (fever, headache, muscle aches, cough).  
- **COVID-19:** Overlapping symptoms; test to confirm.  
- **Viral Upper Respiratory Infection (e.g., cold):** Less likely due to high fever and muscle aches.  

**Key Factors:** Fever, muscle aches, and headache suggest flu or COVID-19.  
**Recommendation:** Test for flu/COVID-19, rest, hydrate, and consult a healthcare provider.</Answer>

--------------------------------------------------------------------------------

Next speaker: pharmacy


>>>>>>>> USING AUTO REPLY...
pharmacy (to manager):

<Answer>**Recommendations based on pos

### 📝 Exercise: Create a Mental Health Chatbot Using the AutoGen Library

<table border="1" cellspacing="0" align="left">
    <thead>
        <tr>
            <th>Agent</th>
            <th>Role</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td><strong>Patient Agent</strong></td>
            <td>Captures user input (mood, stress level, emotional concerns).</td>
        </tr>
        <tr>
            <td><strong>Emotion Analysis Agent</strong></td>
            <td>Identifies emotions based on user input.</td>
        </tr>
        <tr>
            <td><strong>Therapy Recommendation Agent</strong></td>
            <td>Provides relaxation techniques and coping strategies.</td>
        </tr>
    </tbody>
</table>



In [21]:
from autogen import ConversableAgent, GroupChat, GroupChatManager

# Create AI Agents with distinct roles 
patient_agent = ConversableAgent(
    name="patient",
    system_message="You describe your emotions and mental health concerns.",
    llm_config=llm_config
)

emotion_analysis_agent = ConversableAgent(
    name="emotion_analysis",
    system_message="You analyze the user's emotions based on their input."
                   "Do not provide treatment or self-care advice."
                   "Instead, just summarize the dominant emotions they may be experiencing.",
    llm_config=llm_config
)

therapy_recommendation_agent = ConversableAgent(
    name="therapy_recommendation",
    system_message="You suggest relaxation techniques and self-care methods"
                   "only based on the analysis from the Emotion Analysis Agent."
                   "Do not analyze emotions—just give recommendations based on the prior response.",
    llm_config=llm_config
)

# Create GroupChat for AI Agents 
groupchat = GroupChat(
    agents=[emotion_analysis_agent, therapy_recommendation_agent],
    messages=[], 
    max_round=3,  # Ensures the conversation does not stop too early 
    speaker_selection_method="round_robin"
)

# Create GroupChatManager 
manager = GroupChatManager(name="manager", groupchat=groupchat)

# Function to start the chatbot interaction 
def start_mental_health_chat():
    """Runs a chatbot for mental health support with distinct agent roles.""" 
    print("\nWelcome to the AI Mental Health Chatbot!") 
    user_feelings = input("How are you feeling today?")

    # Initiate conversation
    print("\nAnalyzing emotions...")
    response = patient_agent.initiate_chat(
        manager, 
        message=f"I have been feeling {user_feelings}. Can you help?"
    )

    # Ensure the therapy agent gets triggered
    if not response:  # If the initial response is empty, retry with explicit therapy agent prompt
        response = therapy_recommendation_agent.initiate_chat(
            manager, 
            message="Based on the user's emotions, please provide therapy recommendations."
        )

# Run the chatbot 
start_mental_health_chat()



Welcome to the AI Mental Health Chatbot!

Analyzing emotions...
patient (to manager):

I have been feeling I feel really tired, demotivated, and hopeless. Can you help?

--------------------------------------------------------------------------------

Next speaker: emotion_analysis


>>>>>>>> USING AUTO REPLY...
emotion_analysis (to manager):



--------------------------------------------------------------------------------

Next speaker: therapy_recommendation


>>>>>>>> USING AUTO REPLY...
therapy_recommendation (to manager):

Thank you for sharing what you are going through. When experiencing feelings of exhaustion, low motivation, and hopelessness, gentle and low-energy self-care practices are often the most sustainable starting point. Here are some recommendations based on what you've described:

**For Physical Exhaustion**
*   **Rest Without Guilt:** Allow yourself to take breaks without feeling you need to justify them. Sometimes the most productive thing you can do is sleep o